# Proyecto Final de Probabilidades e Inferencia Estadística
## Distance Sampling para estimación de densidad y abundancia

**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958  
**Docente:** Dr. Erick A. Chacón Montalván

### Pregunta central

¿Cómo estimar la densidad y la abundancia real de una población de aves cuando la probabilidad de detección disminuye con la distancia al observador?

Se comparan dos diseños de muestreo aplicados a la misma población de *Winter Wren*:

- **Point transect**
- **Line transect**

La secuencia inferencial es

$$
D_i
\longrightarrow
\widehat\sigma
\longrightarrow
\widehat p
\longrightarrow
\widehat A
\longrightarrow
\widehat N.
$$


## 1. Planteamiento probabilístico

Para un individuo dentro de la región cubierta se define

$$
D=\text{distancia al observador}
$$

y

$$
Z=
\begin{cases}
1,&\text{si el individuo es detectado},\\
0,&\text{si no es detectado}.
\end{cases}
$$

Para un truncamiento máximo $w$,

$$
\Omega=[0,w]\times\{0,1\},
$$

$$
\mathcal F=\mathcal B([0,w])\otimes 2^{\{0,1\}},
$$

y se considera la familia paramétrica

$$
\mathcal P=\{P_\sigma:\sigma>0\}.
$$

La detección se modela mediante

$$
Z\mid D=d
\sim
\operatorname{Bernoulli}(g(d;\sigma)),
$$

donde

$$
g(d;\sigma)
=
P(Z=1\mid D=d)
=
\exp\left(-\frac{d^2}{2\sigma^2}\right).
$$

Como solo se registran los individuos detectados, la variable observada es

$$
D\mid Z=1.
$$


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from io import StringIO
from scipy.optimize import minimize_scalar
from scipy.special import erf


## 2. Datos

Los datos corresponden a la misma zona de estudio de 33.2 ha.

- `wren_5min`: point transect.
- `wren_lt`: line transect.

El cuaderno intenta leer las bases desde GitHub. También contiene una copia interna de respaldo para que la exposición no dependa de la conexión al repositorio.


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
url_punto = "https://raw.githubusercontent.com/jantoniootoya/Probabilidades1/main/wren_5min.csv"
url_linea = "https://raw.githubusercontent.com/jantoniootoya/Probabilidades1/main/wren_lt.csv"

datos_punto = r"""Region.Label,Area,Sample.Label,Effort,object,distance,Study.Area,direction
Montrave,33.2,1,2,5,37,Montrave 1,w
Montrave,33.2,1,2,9,35,Montrave 1,se
Montrave,33.2,1,2,10,75,Montrave 1,sw
Montrave,33.2,10,2,99,120,Montrave 1,ne
Montrave,33.2,10,2,100,40,Montrave 1,s
Montrave,33.2,10,2,101,80,Montrave 1,se
Montrave,33.2,10,2,102,90,Montrave 1,ne
Montrave,33.2,10,2,107,100,Montrave 1,se
Montrave,33.2,11,2,110,80,Montrave 1,ne
Montrave,33.2,11,2,116,68,Montrave 1,se
Montrave,33.2,11,2,117,80,Montrave 1,n
Montrave,33.2,11,2,118,100,Montrave 1,ne
Montrave,33.2,12,2,120,50,Montrave 1,n
Montrave,33.2,12,2,121,80,Montrave 1,ne
Montrave,33.2,12,2,122,88,Montrave 1,w
Montrave,33.2,12,2,123,65,Montrave 1,e
Montrave,33.2,12,2,124,45,Montrave 1,e
Montrave,33.2,12,2,125,45,Montrave 1,sw
Montrave,33.2,12,2,130,50,Montrave 1,n
Montrave,33.2,12,2,131,50,Montrave 1,nw
Montrave,33.2,12,2,132,60,Montrave 1,s
Montrave,33.2,12,2,133,70,Montrave 1,sw
Montrave,33.2,13,2,137,35,Montrave 1,e
Montrave,33.2,13,2,138,55,Montrave 1,w
Montrave,33.2,13,2,139,70,Montrave 1,n
Montrave,33.2,13,2,143,24,Montrave 1,ne
Montrave,33.2,13,2,144,25,Montrave 1,e
Montrave,33.2,13,2,145,45,Montrave 1,nw
Montrave,33.2,13,2,146,90,Montrave 1,w
Montrave,33.2,14,2,151,70,Montrave 1,sw
Montrave,33.2,14,2,152,55,Montrave 1,e
Montrave,33.2,14,2,153,50,Montrave 1,se
Montrave,33.2,14,2,157,45,Montrave 1,s
Montrave,33.2,14,2,158,55,Montrave 1,w
Montrave,33.2,14,2,159,67,Montrave 1,ne
Montrave,33.2,14,2,160,70,Montrave 1,sw
Montrave,33.2,15,2,164,50,Montrave 1,ne
Montrave,33.2,15,2,168,35,Montrave 1,n
Montrave,33.2,15,2,169,60,Montrave 1,nw
Montrave,33.2,16,2,175,32,Montrave 1,sw
Montrave,33.2,16,2,179,55,Montrave 1,e
Montrave,33.2,16,2,180,90,Montrave 1,w
Montrave,33.2,17,2,181,70,Montrave 1,se
Montrave,33.2,17,2,182,30,Montrave 1,ne
Montrave,33.2,17,2,186,40,Montrave 1,se
Montrave,33.2,18,2,188,35,Montrave 1,e
Montrave,33.2,18,2,194,17,Montrave 1,e
Montrave,33.2,18,2,195,30,Montrave 1,se
Montrave,33.2,18,2,196,45,Montrave 1,ne
Montrave,33.2,18,2,197,50,Montrave 1,nw
Montrave,33.2,19,2,201,65,Montrave 1,w
Montrave,33.2,19,2,202,75,Montrave 1,e
Montrave,33.2,19,2,208,35,Montrave 1,nw
Montrave,33.2,2,2,14,40,Montrave 1,e
Montrave,33.2,2,2,18,35,Montrave 1,ne
Montrave,33.2,2,2,19,65,Montrave 1,e
Montrave,33.2,20,2,214,55,Montrave 1,ne
Montrave,33.2,20,2,220,32,Montrave 1,e
Montrave,33.2,20,2,221,85,Montrave 1,ne
Montrave,33.2,21,2,227,50,Montrave 1,w
Montrave,33.2,21,2,230,50,Montrave 1,e
Montrave,33.2,21,2,231,70,Montrave 1,w
Montrave,33.2,22,2,234,45,Montrave 1,s
Montrave,33.2,23,2,245,60,Montrave 1,se
Montrave,33.2,24,2,249,70,Montrave 1,se
Montrave,33.2,24,2,251,44,Montrave 1,w
Montrave,33.2,25,2,258,65,Montrave 1,n
Montrave,33.2,26,2,264,25,Montrave 1,w
Montrave,33.2,26,2,265,45,Montrave 1,n
Montrave,33.2,26,2,266,30,Montrave 1,nw
Montrave,33.2,26,2,271,35,Montrave 1,sw
Montrave,33.2,26,2,272,45,Montrave 1,s
Montrave,33.2,26,2,273,70,Montrave 1,ne
Montrave,33.2,27,2,275,16,Montrave 1,nw
Montrave,33.2,27,2,276,55,Montrave 1,w
Montrave,33.2,27,2,278,4,Montrave 1,n
Montrave,33.2,27,2,279,45,Montrave 1,sw
Montrave,33.2,28,2,287,60,Montrave 1,se
Montrave,33.2,28,2,288,70,Montrave 1,ne
Montrave,33.2,29,2,293,50,Montrave 1,e
Montrave,33.2,29,2,294,45,Montrave 1,nw
Montrave,33.2,29,2,295,21,Montrave 1,w
Montrave,33.2,29,2,296,70,Montrave 1,n
Montrave,33.2,29,2,301,30,Montrave 1,w
Montrave,33.2,29,2,302,35,Montrave 1,s
Montrave,33.2,29,2,303,45,Montrave 1,e
Montrave,33.2,29,2,304,55,Montrave 1,sw
Montrave,33.2,29,2,305,70,Montrave 1,n
Montrave,33.2,3,2,22,62,Montrave 1,e
Montrave,33.2,3,2,23,42,Montrave 1,s
Montrave,33.2,3,2,26,40,Montrave 1,s
Montrave,33.2,3,2,27,45,Montrave 1,w
Montrave,33.2,3,2,28,65,Montrave 1,nw
Montrave,33.2,30,2,309,50,Montrave 1,e
Montrave,33.2,30,2,312,15,Montrave 1,e
Montrave,33.2,30,2,313,25,Montrave 1,sw
Montrave,33.2,30,2,314,50,Montrave 1,se
Montrave,33.2,31,2,318,50,Montrave 1,w
Montrave,33.2,31,2,319,50,Montrave 1,sw
Montrave,33.2,31,2,324,65,Montrave 1,nw
Montrave,33.2,32,2,334,40,Montrave 1,ne
Montrave,33.2,32,2,335,75,Montrave 1,nw
Montrave,33.2,4,2,29,55,Montrave 1,se
Montrave,33.2,4,2,30,45,Montrave 1,ne
Montrave,33.2,4,2,34,42,Montrave 1,e
Montrave,33.2,4,2,35,75,Montrave 1,se
Montrave,33.2,4,2,36,90,Montrave 1,e
Montrave,33.2,4,2,37,120,Montrave 1,w
Montrave,33.2,5,2,43,60,Montrave 1,w
Montrave,33.2,5,2,48,50,Montrave 1,e
Montrave,33.2,5,2,49,55,Montrave 1,w
Montrave,33.2,5,2,50,55,Montrave 1,se
Montrave,33.2,5,2,51,65,Montrave 1,n
Montrave,33.2,6,2,55,35,Montrave 1,sw
Montrave,33.2,6,2,56,33,Montrave 1,s
Montrave,33.2,6,2,62,33,Montrave 1,w
Montrave,33.2,6,2,63,60,Montrave 1,n
Montrave,33.2,7,2,68,82,Montrave 1,sw
Montrave,33.2,7,2,72,55,Montrave 1,ne
Montrave,33.2,7,2,73,55,Montrave 1,nw
Montrave,33.2,7,2,74,75,Montrave 1,ne
Montrave,33.2,7,2,75,85,Montrave 1,se
Montrave,33.2,7,2,76,85,Montrave 1,sw
Montrave,33.2,8,2,80,80,Montrave 1,se
Montrave,33.2,8,2,81,60,Montrave 1,n
Montrave,33.2,8,2,82,90,Montrave 1,nw
Montrave,33.2,8,2,85,50,Montrave 1,n
Montrave,33.2,8,2,86,54,Montrave 1,w
Montrave,33.2,8,2,87,80,Montrave 1,e
Montrave,33.2,9,2,91,31,Montrave 1,w
Montrave,33.2,9,2,92,42,Montrave 1,n
Montrave,33.2,9,2,95,30,Montrave 1,n
Montrave,33.2,9,2,96,40,Montrave 1,sw
Montrave,33.2,9,2,97,45,Montrave 1,e"""
datos_linea = r"""Region.Label,Area,Sample.Label,Effort,object,distance,Study.Area
Montrave,33.2,1,0.416,5,15,Montrave 4
Montrave,33.2,1,0.416,6,80,Montrave 4
Montrave,33.2,1,0.416,7,35,Montrave 4
Montrave,33.2,1,0.416,8,55,Montrave 4
Montrave,33.2,1,0.416,12,12,Montrave 4
Montrave,33.2,1,0.416,13,75,Montrave 4
Montrave,33.2,1,0.416,14,23,Montrave 4
Montrave,33.2,1,0.416,15,70,Montrave 4
Montrave,33.2,1,0.416,16,75,Montrave 4
Montrave,33.2,1,0.416,17,56,Montrave 4
Montrave,33.2,2,0.802,25,20,Montrave 4
Montrave,33.2,2,0.802,26,24,Montrave 4
Montrave,33.2,2,0.802,27,10,Montrave 4
Montrave,33.2,2,0.802,28,60,Montrave 4
Montrave,33.2,2,0.802,29,65,Montrave 4
Montrave,33.2,2,0.802,30,0,Montrave 4
Montrave,33.2,2,0.802,41,40,Montrave 4
Montrave,33.2,2,0.802,42,32,Montrave 4
Montrave,33.2,2,0.802,43,20,Montrave 4
Montrave,33.2,2,0.802,44,60,Montrave 4
Montrave,33.2,2,0.802,45,40,Montrave 4
Montrave,33.2,3,0.802,56,33,Montrave 4
Montrave,33.2,3,0.802,57,5,Montrave 4
Montrave,33.2,3,0.802,58,35,Montrave 4
Montrave,33.2,3,0.802,59,15,Montrave 4
Montrave,33.2,3,0.802,60,70,Montrave 4
Montrave,33.2,3,0.802,61,5,Montrave 4
Montrave,33.2,3,0.802,73,48,Montrave 4
Montrave,33.2,3,0.802,74,32,Montrave 4
Montrave,33.2,3,0.802,75,0,Montrave 4
Montrave,33.2,3,0.802,76,33,Montrave 4
Montrave,33.2,3,0.802,77,20,Montrave 4
Montrave,33.2,3,0.802,78,35,Montrave 4
Montrave,33.2,4,0.598,87,5,Montrave 4
Montrave,33.2,4,0.598,88,80,Montrave 4
Montrave,33.2,4,0.598,89,80,Montrave 4
Montrave,33.2,4,0.598,90,55,Montrave 4
Montrave,33.2,4,0.598,98,70,Montrave 4
Montrave,33.2,4,0.598,99,75,Montrave 4
Montrave,33.2,4,0.598,100,22,Montrave 4
Montrave,33.2,4,0.598,101,85,Montrave 4
Montrave,33.2,4,0.598,102,75,Montrave 4
Montrave,33.2,4,0.598,103,35,Montrave 4
Montrave,33.2,4,0.598,104,40,Montrave 4
Montrave,33.2,5,0.7,115,35,Montrave 4
Montrave,33.2,5,0.7,116,10,Montrave 4
Montrave,33.2,5,0.7,117,70,Montrave 4
Montrave,33.2,5,0.7,118,70,Montrave 4
Montrave,33.2,5,0.7,119,75,Montrave 4
Montrave,33.2,5,0.7,132,60,Montrave 4
Montrave,33.2,5,0.7,133,30,Montrave 4
Montrave,33.2,5,0.7,134,25,Montrave 4
Montrave,33.2,5,0.7,135,80,Montrave 4
Montrave,33.2,5,0.7,136,80,Montrave 4
Montrave,33.2,5,0.7,137,40,Montrave 4
Montrave,33.2,5,0.7,138,85,Montrave 4
Montrave,33.2,5,0.7,139,55,Montrave 4
Montrave,33.2,5,0.7,140,65,Montrave 4
Montrave,33.2,6,0.802,149,20,Montrave 4
Montrave,33.2,6,0.802,150,30,Montrave 4
Montrave,33.2,6,0.802,151,40,Montrave 4
Montrave,33.2,6,0.802,152,25,Montrave 4
Montrave,33.2,6,0.802,153,45,Montrave 4
Montrave,33.2,6,0.802,154,60,Montrave 4
Montrave,33.2,6,0.802,162,25,Montrave 4
Montrave,33.2,6,0.802,163,35,Montrave 4
Montrave,33.2,6,0.802,164,25,Montrave 4
Montrave,33.2,6,0.802,165,10,Montrave 4
Montrave,33.2,6,0.802,166,2,Montrave 4
Montrave,33.2,6,0.802,167,35,Montrave 4
Montrave,33.2,7,0.786,174,27,Montrave 4
Montrave,33.2,7,0.786,175,35,Montrave 4
Montrave,33.2,7,0.786,176,65,Montrave 4
Montrave,33.2,7,0.786,177,45,Montrave 4
Montrave,33.2,7,0.786,178,80,Montrave 4
Montrave,33.2,7,0.786,179,30,Montrave 4
Montrave,33.2,7,0.786,187,25,Montrave 4
Montrave,33.2,7,0.786,188,70,Montrave 4
Montrave,33.2,7,0.786,189,30,Montrave 4
Montrave,33.2,7,0.786,190,55,Montrave 4
Montrave,33.2,7,0.786,191,60,Montrave 4
Montrave,33.2,7,0.786,192,40,Montrave 4
Montrave,33.2,7,0.786,193,75,Montrave 4
Montrave,33.2,8,0.81,199,70,Montrave 4
Montrave,33.2,8,0.81,200,23,Montrave 4
Montrave,33.2,8,0.81,201,55,Montrave 4
Montrave,33.2,8,0.81,202,60,Montrave 4
Montrave,33.2,8,0.81,203,75,Montrave 4
Montrave,33.2,8,0.81,204,30,Montrave 4
Montrave,33.2,8,0.81,205,65,Montrave 4
Montrave,33.2,8,0.81,211,45,Montrave 4
Montrave,33.2,8,0.81,212,20,Montrave 4
Montrave,33.2,8,0.81,213,55,Montrave 4
Montrave,33.2,8,0.81,214,30,Montrave 4
Montrave,33.2,9,0.77,217,45,Montrave 4
Montrave,33.2,9,0.77,218,10,Montrave 4
Montrave,33.2,9,0.77,219,75,Montrave 4
Montrave,33.2,9,0.77,223,30,Montrave 4
Montrave,33.2,9,0.77,224,70,Montrave 4
Montrave,33.2,9,0.77,225,35,Montrave 4
Montrave,33.2,10,0.408,229,55,Montrave 4
Montrave,33.2,10,0.408,230,23,Montrave 4
Montrave,33.2,10,0.408,233,85,Montrave 4
Montrave,33.2,11,0.078,234,25,Montrave 4
Montrave,33.2,12,0.094,237,90,Montrave 4
Montrave,33.2,12,0.094,238,40,Montrave 4
Montrave,33.2,12,0.094,240,50,Montrave 4
Montrave,33.2,12,0.094,241,90,Montrave 4
Montrave,33.2,13,0.408,246,40,Montrave 4
Montrave,33.2,13,0.408,247,0,Montrave 4
Montrave,33.2,13,0.408,248,100,Montrave 4
Montrave,33.2,13,0.408,249,45,Montrave 4
Montrave,33.2,13,0.408,250,20,Montrave 4
Montrave,33.2,13,0.408,256,40,Montrave 4
Montrave,33.2,13,0.408,257,20,Montrave 4
Montrave,33.2,13,0.408,258,45,Montrave 4
Montrave,33.2,13,0.408,259,45,Montrave 4
Montrave,33.2,14,0.542,265,80,Montrave 4
Montrave,33.2,14,0.542,266,5,Montrave 4
Montrave,33.2,14,0.542,267,35,Montrave 4
Montrave,33.2,14,0.542,268,35,Montrave 4
Montrave,33.2,14,0.542,269,10,Montrave 4
Montrave,33.2,14,0.542,273,50,Montrave 4
Montrave,33.2,14,0.542,274,30,Montrave 4
Montrave,33.2,14,0.542,275,20,Montrave 4
Montrave,33.2,14,0.542,276,35,Montrave 4
Montrave,33.2,14,0.542,277,60,Montrave 4
Montrave,33.2,14,0.542,278,15,Montrave 4
Montrave,33.2,14,0.542,279,10,Montrave 4
Montrave,33.2,15,0.472,286,22,Montrave 4
Montrave,33.2,15,0.472,287,15,Montrave 4
Montrave,33.2,15,0.472,288,30,Montrave 4
Montrave,33.2,15,0.472,289,40,Montrave 4
Montrave,33.2,15,0.472,294,15,Montrave 4
Montrave,33.2,15,0.472,295,35,Montrave 4
Montrave,33.2,15,0.472,296,16,Montrave 4
Montrave,33.2,15,0.472,297,15,Montrave 4
Montrave,33.2,15,0.472,298,25,Montrave 4
Montrave,33.2,16,0.378,301,60,Montrave 4
Montrave,33.2,16,0.378,303,35,Montrave 4
Montrave,33.2,17,0.354,312,10,Montrave 4
Montrave,33.2,17,0.354,313,55,Montrave 4
Montrave,33.2,17,0.354,314,12,Montrave 4
Montrave,33.2,17,0.354,318,35,Montrave 4
Montrave,33.2,17,0.354,319,30,Montrave 4
Montrave,33.2,17,0.354,320,70,Montrave 4
Montrave,33.2,17,0.354,321,25,Montrave 4
Montrave,33.2,18,0.4,326,10,Montrave 4
Montrave,33.2,18,0.4,327,60,Montrave 4
Montrave,33.2,18,0.4,335,10,Montrave 4
Montrave,33.2,18,0.4,336,60,Montrave 4
Montrave,33.2,18,0.4,337,40,Montrave 4
Montrave,33.2,18,0.4,338,50,Montrave 4
Montrave,33.2,19,0.04,340,40,Montrave 4
Montrave,33.2,19,0.04,341,80,Montrave 4
Montrave,33.2,19,0.04,343,25,Montrave 4"""

try:
    punto = pd.read_csv(url_punto)
    linea = pd.read_csv(url_linea)
except:
    punto = pd.read_csv(StringIO(datos_punto))
    linea = pd.read_csv(StringIO(datos_linea))

punto.shape, linea.shape


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
display(punto.head())
display(linea.head())


## 3. Preparación del esfuerzo

Para point transect:

$$
S_P=K\pi w_P^2.
$$

Para line transect:

$$
S_L=2w_LL.
$$

Las áreas se expresan en hectáreas.


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
dP = punto["distance"].to_numpy(dtype=float)
dL = linea["distance"].to_numpy(dtype=float)

nP = len(dP)
nL = len(dL)

wP = dP.max()
wL = dL.max()

K = punto[["Sample.Label", "Effort"]].drop_duplicates()["Effort"].sum()
L = linea[["Sample.Label", "Effort"]].drop_duplicates()["Effort"].sum()

area_estudio = float(punto["Area"].iloc[0])

S_P = K * np.pi * wP**2 / 10000
S_L = 2 * wL * (1000 * L) / 10000


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
resumen = pd.DataFrame({
    "Diseño": ["Point transect", "Line transect"],
    "Detecciones": [nP, nL],
    "Unidades": [punto["Sample.Label"].nunique(), linea["Sample.Label"].nunique()],
    "Esfuerzo": [K, L],
    "w (m)": [wP, wL],
    "Área cubierta (ha)": [S_P, S_L],
    "Área de estudio (ha)": [area_estudio, area_estudio]
})

resumen.round(4)


### Lectura de los datos

Los valores esperados son:

- Point transect: 134 detecciones, 32 puntos, 64 visitas acumuladas, $w_P=120$ m.
- Line transect: 156 detecciones, 19 transectos, $L=9.66$ km, $w_L=100$ m.


## 4. Análisis exploratorio

La forma de las distancias observadas depende de dos elementos:

1. la geometría del diseño;
2. la pérdida de detectabilidad con la distancia.

En line transect, la cantidad de área disponible por franja de distancia es constante.

En point transect, el área disponible aumenta con el radio.


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(dP, bins=12)
plt.xlabel("Distancia radial (m)")
plt.ylabel("Frecuencia")
plt.title("Point transect")
plt.show()


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(dL, bins=12)
plt.xlabel("Distancia perpendicular (m)")
plt.ylabel("Frecuencia")
plt.title("Line transect")
plt.show()


## 5. Geometría de los diseños

### Line transect

$$
P(D\le d)=\frac{d}{w},
\qquad
\pi_L(d)=\frac{1}{w}.
$$

### Point transect

$$
P(D\le d)=\frac{d^2}{w^2},
\qquad
\pi_P(d)=\frac{2d}{w^2}.
$$

Esta diferencia geométrica explica por qué los histogramas no deben tener la misma forma aun cuando la función de detección pertenezca a la misma familia.


## 6. Probabilidad promedio de detección

La probabilidad promedio de detección es

$$
p(\sigma)
=
P(Z=1)
=
\int_0^w \pi(d)g(d;\sigma)\,dd.
$$

Para line transect:

$$
p_L(\sigma)
=
\frac{\sigma}{w}
\sqrt{\frac{\pi}{2}}
\operatorname{erf}
\left(
\frac{w}{\sqrt{2}\sigma}
\right).
$$

Para point transect:

$$
p_P(\sigma)
=
\frac{2\sigma^2}{w^2}
\left[
1-
\exp\left(
-\frac{w^2}{2\sigma^2}
\right)
\right].
$$


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
def g(d, sigma):
    return np.exp(-(d**2) / (2 * sigma**2))

def pP(sigma):
    return (2 * sigma**2 / wP**2) * (1 - np.exp(-(wP**2) / (2 * sigma**2)))

def pL(sigma):
    return (sigma / wL) * np.sqrt(np.pi / 2) * erf(wL / (np.sqrt(2) * sigma))


## 7. Distribución de las distancias detectadas

Por probabilidad condicional,

$$
f(d\mid Z=1;\sigma)
=
\frac{\pi(d)g(d;\sigma)}
{p(\sigma)}.
$$

Para line transect:

$$
f_L(d\mid Z=1;\sigma)
=
\frac{
\exp\left(-d^2/(2\sigma^2)\right)
}{
\sigma\sqrt{\pi/2}
\operatorname{erf}
\left(w/(\sqrt{2}\sigma)\right)
}.
$$

Para point transect:

$$
f_P(d\mid Z=1;\sigma)
=
\frac{
d\exp\left(-d^2/(2\sigma^2)\right)
}{
\sigma^2
\left[
1-\exp\left(-w^2/(2\sigma^2)\right)
\right]
}.
$$


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
def fP(d, sigma):
    return d * np.exp(-(d**2) / (2 * sigma**2)) / (
        sigma**2 * (1 - np.exp(-(wP**2) / (2 * sigma**2)))
    )

def fL(d, sigma):
    return np.exp(-(d**2) / (2 * sigma**2)) / (
        sigma * np.sqrt(np.pi / 2) * erf(wL / (np.sqrt(2) * sigma))
    )


## 8. Máxima verosimilitud

Para las distancias observadas

$$
d_1,\ldots,d_n,
$$

la verosimilitud condicional es

$$
L(\sigma)
=
\prod_{i=1}^{n}
f(d_i\mid Z=1;\sigma).
$$

Se estima

$$
\widehat\sigma
=
\arg\max_{\sigma>0}\ell(\sigma).
$$


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
def loglikP(sigma):
    return (
        np.log(dP).sum()
        - 2 * nP * np.log(sigma)
        - (dP**2).sum() / (2 * sigma**2)
        - nP * np.log(1 - np.exp(-(wP**2) / (2 * sigma**2)))
    )

def loglikL(sigma):
    return (
        -(dL**2).sum() / (2 * sigma**2)
        - nL * np.log(
            sigma * np.sqrt(np.pi / 2) * erf(wL / (np.sqrt(2) * sigma))
        )
    )


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
ajuste_P = minimize_scalar(lambda sigma: -loglikP(sigma), bounds=(0.1, 500), method="bounded")
ajuste_L = minimize_scalar(lambda sigma: -loglikL(sigma), bounds=(0.1, 500), method="bounded")

sigma_P = ajuste_P.x
sigma_L = ajuste_L.x

pd.DataFrame({
    "Diseño": ["Point transect", "Line transect"],
    "sigma estimado": [sigma_P, sigma_L]
}).round(6)


## 9. Detección, densidad y abundancia

Una vez estimado $\sigma$:

$$
\widehat p=p(\widehat\sigma).
$$

La densidad se obtiene mediante

$$
\widehat A
=
\frac{n}{S\widehat p},
$$

y la abundancia mediante

$$
\widehat N
=
\widehat A A_{\mathrm{study}}.
$$


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
p_det_P = pP(sigma_P)
p_det_L = pL(sigma_L)

A_P = nP / (S_P * p_det_P)
A_L = nL / (S_L * p_det_L)

N_P = A_P * area_estudio
N_L = A_L * area_estudio

estimaciones = pd.DataFrame({
    "Diseño": ["Point transect", "Line transect"],
    "sigma": [sigma_P, sigma_L],
    "p detección": [p_det_P, p_det_L],
    "Densidad (aves/ha)": [A_P, A_L],
    "Abundancia": [N_P, N_L]
})

estimaciones.round(6)


### Interpretación de los estimadores puntuales

- **Point transect:** menor probabilidad promedio de detección y mayor corrección del conteo observado.
- **Line transect:** mayor probabilidad promedio de detección y menor corrección.
- Los dos diseños estudian la misma población, pero no deben producir necesariamente el mismo $\widehat\sigma$ porque la geometría de observación es diferente.


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
xP = np.linspace(0, wP, 400)

plt.figure(figsize=(8, 5))
plt.plot(xP, g(xP, sigma_P))
plt.xlabel("Distancia (m)")
plt.ylabel("Probabilidad de detección")
plt.ylim(0, 1)
plt.title("Función de detección estimada — Point transect")
plt.show()


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
xL = np.linspace(0, wL, 400)

plt.figure(figsize=(8, 5))
plt.plot(xL, g(xL, sigma_L))
plt.xlabel("Distancia (m)")
plt.ylabel("Probabilidad de detección")
plt.ylim(0, 1)
plt.title("Función de detección estimada — Line transect")
plt.show()


## 10. Comprobación visual del ajuste

Se compara el histograma normalizado con la densidad estimada de

$$
D\mid Z=1.
$$

El objetivo no es demostrar ajuste perfecto, sino verificar si la forma teórica reproduce razonablemente el patrón empírico.


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(dP, bins=12, density=True, alpha=0.5)
plt.plot(xP[1:], fP(xP[1:], sigma_P))
plt.xlabel("Distancia radial (m)")
plt.ylabel("Densidad")
plt.title("Ajuste half-normal — Point transect")
plt.show()


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(dL, bins=12, density=True, alpha=0.5)
plt.plot(xL, fL(xL, sigma_L))
plt.xlabel("Distancia perpendicular (m)")
plt.ylabel("Densidad")
plt.title("Ajuste half-normal — Line transect")
plt.show()


## 11. Cuantificación de incertidumbre

Para incorporar la incertidumbre de la densidad se hace explícito un supuesto adicional:

$$
N(S)\sim\operatorname{Poisson}(AS).
$$

Después del mecanismo de detección,

$$
n\sim\operatorname{Poisson}(ASp(\sigma)).
$$

La log-verosimilitud conjunta es

$$
\ell(A,\sigma)
=
n\log(ASp(\sigma))
-
ASp(\sigma)
+
\sum_{i=1}^{n}
\log f(d_i\mid Z=1;\sigma).
$$

Se parametriza

$$
\eta_A=\log A,
\qquad
\eta_\sigma=\log\sigma,
$$

y se utiliza la información observada

$$
J(\widehat\eta)
=
-\nabla^2\ell(\widehat\eta).
$$

Los intervalos reportados son **intervalos asintóticos basados en la información observada del modelo Poisson–distance sampling**.

No se utiliza bootstrap.


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
def loglik_conjunta_P(theta):
    A, sigma = np.exp(theta)
    p = pP(sigma)
    return nP * np.log(A * S_P * p) - A * S_P * p + loglikP(sigma)

def loglik_conjunta_L(theta):
    A, sigma = np.exp(theta)
    p = pL(sigma)
    return nL * np.log(A * S_L * p) - A * S_L * p + loglikL(sigma)


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
def hessiano(f, x, h=1e-4):
    x = np.asarray(x, dtype=float)
    m = len(x)
    H = np.zeros((m, m))
    fx = f(x)

    for i in range(m):
        ei = np.zeros(m)
        ei[i] = h
        H[i, i] = (f(x + ei) - 2 * fx + f(x - ei)) / h**2

        for j in range(i + 1, m):
            ej = np.zeros(m)
            ej[j] = h
            H[i, j] = (
                f(x + ei + ej)
                - f(x + ei - ej)
                - f(x - ei + ej)
                + f(x - ei - ej)
            ) / (4 * h**2)
            H[j, i] = H[i, j]

    return H


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
theta_P = np.log([A_P, sigma_P])
theta_L = np.log([A_L, sigma_L])

V_P = np.linalg.inv(-hessiano(loglik_conjunta_P, theta_P))
V_L = np.linalg.inv(-hessiano(loglik_conjunta_L, theta_L))

se_P = np.sqrt(np.diag(V_P))
se_L = np.sqrt(np.diag(V_L))

IC_A_P = np.exp([theta_P[0] - 1.96 * se_P[0], theta_P[0] + 1.96 * se_P[0]])
IC_A_L = np.exp([theta_L[0] - 1.96 * se_L[0], theta_L[0] + 1.96 * se_L[0]])

IC_sigma_P = np.exp([theta_P[1] - 1.96 * se_P[1], theta_P[1] + 1.96 * se_P[1]])
IC_sigma_L = np.exp([theta_L[1] - 1.96 * se_L[1], theta_L[1] + 1.96 * se_L[1]])

IC_p_P = pP(IC_sigma_P)
IC_p_L = pL(IC_sigma_L)

IC_N_P = IC_A_P * area_estudio
IC_N_L = IC_A_L * area_estudio


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
resultados = pd.DataFrame({
    "Diseño": ["Point transect", "Line transect"],
    "sigma": [sigma_P, sigma_L],
    "IC95 sigma": [
        f"[{IC_sigma_P[0]:.4f}, {IC_sigma_P[1]:.4f}]",
        f"[{IC_sigma_L[0]:.4f}, {IC_sigma_L[1]:.4f}]"
    ],
    "p": [p_det_P, p_det_L],
    "IC95 p": [
        f"[{IC_p_P[0]:.4f}, {IC_p_P[1]:.4f}]",
        f"[{IC_p_L[0]:.4f}, {IC_p_L[1]:.4f}]"
    ],
    "Densidad": [A_P, A_L],
    "IC95 densidad": [
        f"[{IC_A_P[0]:.4f}, {IC_A_P[1]:.4f}]",
        f"[{IC_A_L[0]:.4f}, {IC_A_L[1]:.4f}]"
    ],
    "Abundancia": [N_P, N_L],
    "IC95 abundancia": [
        f"[{IC_N_P[0]:.2f}, {IC_N_P[1]:.2f}]",
        f"[{IC_N_L[0]:.2f}, {IC_N_L[1]:.2f}]"
    ]
})

resultados.round({"sigma":4, "p":4, "Densidad":4, "Abundancia":2})


**José Antonio Otoya Barrenechea**  
**Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería**  
**ORCID:** 0009-0007-0702-6958


In [ ]:
densidades = np.array([A_P, A_L])
errores_inf = densidades - np.array([IC_A_P[0], IC_A_L[0]])
errores_sup = np.array([IC_A_P[1], IC_A_L[1]]) - densidades

plt.figure(figsize=(8, 5))
plt.errorbar(
    ["Point transect", "Line transect"],
    densidades,
    yerr=np.vstack([errores_inf, errores_sup]),
    fmt="o",
    capsize=6
)
plt.ylabel("Densidad estimada (aves/ha)")
plt.title("Densidad e IC 95% por diseño")
plt.show()


## 12. Validación externa del line transect

El ajuste oficial del paquete `Distance`, para los mismos 156 registros y una half-normal sin ajustes, reporta aproximadamente:

$$
\widehat\sigma_L=60.6923,
$$

$$
\widehat p_L=0.685037,
$$

$$
\widehat A_L=1.1787\;\text{aves/ha},
$$

$$
\widehat N_L=39.1329.
$$

Los resultados obtenidos en este cuaderno reproducen esos valores prácticamente de forma exacta.

**Referencia de validación:** documentación oficial del paquete `Distance`, ejemplo `lines-distill`.

https://distancesampling.org/Distance/articles/lines-distill.html


## 13. Comparación e interpretación final

Los resultados principales son aproximadamente:

| Diseño | $\widehat\sigma$ | $\widehat p$ | $\widehat A$ (aves/ha) | $\widehat N$ |
|---|---:|---:|---:|---:|
| Point transect | 43.3746 | 0.2556 | 1.8107 | 60.11 |
| Line transect | 60.6923 | 0.6850 | 1.1787 | 39.13 |

Los intervalos de confianza de densidad presentan una **pequeña superposición**. Esto indica compatibilidad descriptiva parcial entre los resultados, pero la superposición de intervalos **no constituye una prueba formal de igualdad** entre ambos estimadores.

La diferencia entre las estimaciones puede estar asociada a la geometría de los diseños, al patrón de detección y a las limitaciones de la función half-normal.


## 14. Limitaciones

1. La función half-normal se adopta como modelo base; no se afirma que sea la única ni la mejor función posible.
2. Para point transect se utiliza $w_P=120$ m, correspondiente a la máxima distancia incluida en la base.
3. Los datos de point transect presentan discretización o redondeo de distancias, mientras que la formulación desarrollada es continua.
4. Para la incertidumbre de la densidad se introduce explícitamente un proceso espacial homogéneo de Poisson.
5. Los intervalos obtenidos no son idénticos a los intervalos del software `Distance`, porque el procedimiento de varianza no es exactamente el mismo.
6. En line transect el patrón empírico muestra concentración de detecciones lejos de cero; por ello, otras funciones de detección podrían mejorar el ajuste.


## 15. Conclusiones

1. El mecanismo de observación puede formalizarse mediante $D$ y $Z$, con

$$
Z\mid D=d
\sim
\operatorname{Bernoulli}(g(d;\sigma)).
$$

2. La geometría del diseño determina la distribución previa de las distancias y obliga a utilizar modelos diferentes para point y line transect.

3. La máxima verosimilitud permite estimar la escala de detección:

$$
\widehat\sigma_P\approx43.37,
\qquad
\widehat\sigma_L\approx60.69.
$$

4. Las probabilidades promedio estimadas de detección son:

$$
\widehat p_P\approx0.256,
\qquad
\widehat p_L\approx0.685.
$$

5. Las densidades estimadas son:

$$
\widehat A_P\approx1.811\;\text{aves/ha},
$$

$$
\widehat A_L\approx1.179\;\text{aves/ha}.
$$

6. Las abundancias estimadas en las 33.2 ha son aproximadamente:

$$
\widehat N_P\approx60.11,
\qquad
\widehat N_L\approx39.13.
$$

7. El line transect reproduce prácticamente de manera exacta los resultados de referencia del paquete `Distance`, lo que valida la implementación de máxima verosimilitud utilizada en el cuaderno.


## 16. Esquema para la exposición

### Problema
Los conteos observados subestiman la población porque la detección disminuye con la distancia.

### Modelo
Se separa la ubicación del individuo, $D$, del mecanismo de detección, $Z\mid D$.

### Diferencia entre diseños
La geometría de point transect y line transect produce diferentes distribuciones de distancia.

### Inferencia
Se estima $\sigma$ mediante máxima verosimilitud.

### Corrección
Con $\widehat p$ se corrige el conteo y se obtienen densidad y abundancia.

### Incertidumbre
Se utiliza información observada bajo un supuesto Poisson explícito.

### Resultado principal
El line transect reproduce los valores oficiales de referencia y ambos diseños producen estimaciones de densidad del mismo orden de magnitud, aunque no idénticas.


## Referencias

Buckland, S. T. (2006). Point-Transect Surveys for Songbirds: Robust Methodologies. *The Auk, 123*(2), 345–357.

Buckland, S. T., Anderson, D. R., Burnham, K. P., Laake, J. L., Borchers, D. L., & Thomas, L. (2001). *Introduction to Distance Sampling: Estimating Abundance of Biological Populations*. Oxford University Press.

Thomas, L., Buckland, S. T., Rexstad, E. A., et al. (2010). Distance software: design and analysis of distance sampling surveys for estimating population size. *Journal of Applied Ecology, 47*(1), 5–14.

Distance Development Team. `Distance` documentation: line transect example.  
https://distancesampling.org/Distance/articles/lines-distill.html
